In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.txt', sep = ';', header = None, names = ['text', 'emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [5]:
df.isnull().sum()

,0
text,0
emotion,0


In [6]:
df['emotion'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

**Converting output variable into numeric term**

In [7]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [8]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


**Lowercasing**

In [9]:
df['text'] = df['text'].apply(lambda x: x.lower())

**Removing Punctuation**

In [10]:
import string
def remove_func(txt):
  return txt.translate(str.maketrans('', '', string.punctuation))

In [11]:
df['text'] = df['text'].apply(remove_func)

**Remove Number**

In [12]:
def remove_num(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new

df['text'] = df['text'].apply(remove_num)

**Remove Links**

In [13]:
# import re

In [14]:
# def remove_urls(text):
#     url_pattern = re.compile(r'https?://\S+|www\.\S+')
#     return url_pattern.sub(r'', text)

# df['text'] = df['text'].apply(remove_urls)

**Removing HTML Tags**

In [15]:
# text = """<html><div>
# <h1>Natural Language Procesiong</h1>
# <p>Text Preprocessing for NLP</p>
# <a href="https://abc.com/@def">ABC Account</a>
# </div></html>
# """

# html_tags_pattern = r'<.*?>'

# # replace with empty where html_tags_pattern found
# text_without_html_tags = re.sub(html_tags_pattern, '', text)
# print(text_without_html_tags)

**Remove emojis and Special Characters**

In [17]:
def remove_emoji(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emoji)

**Remove stopwords e.g. is, the, was, in (Remove in ML context but keep in DL context)**

In [18]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [23]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
stop_words = set(stopwords.words('english'))

In [25]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [26]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

df['text'] = df['text'].apply(remove)

In [27]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [28]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


**Bag of Words**

In [29]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "I love pizza",
    "Pizza is the best",
    "I love pasta",
    "Pasta is great"
]

# trigram only 3
vectorizer = CountVectorizer(ngram_range = (3, 3))

X = vectorizer.fit_transform(documents)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("\nBoW Matrix:\n", X.toarray())

Vocabulary: ['is the best' 'pasta is great' 'pizza is the']

BoW Matrix:
 [[0 0 0]
 [1 0 1]
 [0 0 0]
 [0 1 0]]


**TF-IDF**

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "I love pizza",
    "Pizza is the best",
    "I love pasta",
    "Pasta is great"
]

# trigram only 3
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(documents)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("\nBoW Matrix:\n", X.toarray())

Vocabulary: ['best' 'great' 'is' 'love' 'pasta' 'pizza' 'the']

BoW Matrix:
 [[0.         0.         0.         0.70710678 0.         0.70710678
  0.        ]
 [0.55528266 0.         0.43779123 0.         0.         0.43779123
  0.55528266]
 [0.         0.         0.         0.70710678 0.70710678 0.
  0.        ]
 [0.         0.66767854 0.52640543 0.         0.52640543 0.
  0.        ]]


**Project Continuation after df.head()**

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [32]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


In [33]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [34]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [35]:
y_pred = nb2_model.predict(X_test_tfidf)

In [36]:
print(accuracy_score(y_test, y_pred))

0.6609375


In [37]:
from sklearn.linear_model import LogisticRegression

In [38]:
logistic_model = LogisticRegression(max_iter=1000)

In [39]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [40]:
log_pred = logistic_model.predict(X_test_tfidf)

In [41]:
print(accuracy_score(y_test,log_pred ))

0.8628125
